# 💸 Financial Impulse Behaviour Analyser
### Behavioural Analytics Hackathon — Problem Statement 2

---

This notebook is the **reference implementation** of the `BankStatementAnalyser` module.

The core logic lives in `bank_analyser/analyser.py` — a self-contained Python module
designed to be plugged into any web backend (Flask, FastAPI, Django).

| Capability | Detail |
|---|---|
| **Dynamic column detection** | Works with any bank's CSV/TSV/XLSX export — no hardcoding |
| **Format support** | Dual-column (debit/credit), single-amount+flag, signed-amount |
| **Risk scoring** | ML ensemble: Z-Score anomaly + Isolation Forest + PCA reconstruction error |
| **Adaptive thresholds** | Low/Medium/High set from user's own percentile distribution |
| **Output** | JSON-serialisable dict with charts as base64 PNGs — plug directly into any frontend |


## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '.')   # make sure bank_analyser/ is on the path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import base64, io, json
import warnings
warnings.filterwarnings('ignore')

# ── Import the module ──────────────────────────────────────────────────────
from bank_analyser import BankStatementAnalyser, COLUMN_SEMANTIC_MAP, CATEGORY_RULES

print('✅ BankStatementAnalyser loaded.')
print(f'   Supports {len(COLUMN_SEMANTIC_MAP)} canonical column roles')
print(f'   Supports {len(CATEGORY_RULES)} spending categories')


## 2. Dynamic Column Detection — How It Works

Instead of hardcoding `'Withdrawal Amt.'`, `'Chq./Ref.No.'` etc., the module uses a
**semantic mapping** of regex patterns → canonical names.

This means it works out-of-the-box for:
- IndusInd Bank (`Withdrawal Amt.`, `Deposit Amt.`, `Closing Balance`)
- HDFC Bank (`Debit`, `Credit`, `Balance`)
- ICICI Bank (`Withdrawal Amount (INR)`, `Deposit Amount (INR)`)
- SBI (`Debit`, `Credit`, `Balance`)
- Axis Bank (`Debit`, `Credit`, `Balance`)
- Any bank with a single `Amount` + `Dr/Cr` flag column
- Any bank with a signed `Amount` column (negative = debit)

To add a new bank format, just add a regex pattern to `COLUMN_SEMANTIC_MAP` in `analyser.py`.


In [ ]:
# Show the full semantic mapping
print("=" * 65)
print("  CANONICAL ROLE → ACCEPTED COLUMN NAME PATTERNS")
print("=" * 65)
for role, patterns in COLUMN_SEMANTIC_MAP.items():
    print(f"\n  {role.upper()}")
    for p in patterns:
        print(f"    • {p}")


## 3. Run Analysis on Your Bank Statement

Set `FILE_PATH` to your uploaded file. The analyser auto-detects format, delimiter,
column names, and amount structure.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# SET YOUR FILE PATH HERE
# Supported: .csv  .tsv  .txt  .xlsx  .xls
# ─────────────────────────────────────────────────────────────────
FILE_PATH = 'combined_bank_statements.csv'   # ← change this

analyser = BankStatementAnalyser(currency_symbol='₹')

with open(FILE_PATH, 'rb') as f:
    file_bytes = f.read()

result = analyser.analyse(file_bytes=file_bytes, filename=FILE_PATH)

print('✅ Analysis complete.')
print(f'   Date range    : {result["summary"]["date_range"]}')
print(f'   Transactions  : {result["summary"]["total_transactions"]:,}')
print(f'   Months analysed: {result["summary"]["total_months"]}')


## 4. Schema Detection Report

See exactly which columns were detected and how.

In [ ]:
schema = result['schema_report']

print("=" * 65)
print("  SCHEMA DETECTION REPORT")
print("=" * 65)
print(f"\n  Raw columns in file   : {schema['raw_columns']}")
print(f"  Amount format         : {schema['amount_format']}")
print(f"  Rows loaded           : {schema['total_rows_loaded']:,}")
print(f"\n  {'Canonical Role':<22} → {'Mapped to Raw Column'}")
print("  " + "-" * 45)
for role, raw_col in schema['detected_mapping'].items():
    print(f"  {role:<22} → {raw_col}")

if schema['warnings']:
    print(f"\n  ⚠️  Warnings ({len(schema['warnings'])}):")
    for w in schema['warnings']:
        print(f"    • {w}")
else:
    print("\n  ✅ No warnings — clean detection.")


## 5. Data-Driven Risk Scores

Three ML/statistical methods, ensembled into a single score.
Thresholds are **adaptive** — derived from your own spending distribution.


In [ ]:
summary = result['summary']
thresholds = summary['adaptive_thresholds']

print(f"  Adaptive Thresholds (your data):")
print(f"    Low    : score < {thresholds['low_below']}")
print(f"    Medium : {thresholds['medium_range']}")
print(f"    High   : score ≥ {thresholds['high_above']}")
print()

print(f"  Risk Distribution: {result['risk_distribution']}")
print()

# Monthly breakdown table
monthly = result['monthly_scores']
print(f"{'Month':<10} {'Z-Sc':>7} {'IsoF':>7} {'PCA':>7} {'Ensemble':>9} {'Tier':<10} Profile")
print("-" * 80)
for m in monthly:
    print(f"{m['month']:<10} {m['score_zscore']:>7.1f} {m['score_isoforest']:>7.1f}"
          f" {m['score_pca']:>7.1f} {m['impulse_risk_score']:>9.1f}"
          f" {m['risk_tier']:<10} {m['behaviour_profile']}")


## 6. Personalised Behavioural Nudges

In [ ]:
print("=" * 65)
print("  PERSONALISED NUDGES — BASED ON YOUR SPENDING PATTERNS")
print("=" * 65)
for m in monthly:
    print(f"\n📅 {m['month']}  |  Score: {m['impulse_risk_score']:.1f}  |  {m['risk_tier']} Risk")
    print(f"   Profile : {m['behaviour_profile']}")
    for nudge in m['nudges']:
        print(f"   • {nudge}")


## 7. Spending by Category

In [ ]:
from bank_analyser import IMPULSIVE_CATEGORIES

print("\n  Category Spend Breakdown:")
print(f"  {'Category':<30} {'Total':>12}  Flag")
print("  " + "-" * 55)
for cat, amt in result['category_spend'].items():
    flag = '🔴 IMPULSIVE' if cat in IMPULSIVE_CATEGORIES else '🔵'
    print(f"  {cat:<30} ₹{amt:>10,.2f}  {flag}")


## 8. Analytics Charts

All charts are returned as base64-encoded PNGs — paste directly into an `<img>` tag in your frontend.


In [ ]:
def show_chart(b64_str, title=''):
    img_bytes = base64.b64decode(b64_str)
    img = mpimg.imread(io.BytesIO(img_bytes), format='png')
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(img); ax.axis('off')
    if title: ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

charts = result['charts']

for chart_name in ['risk_timeline', 'method_breakdown', 'monthly_trend',
                   'category_spend', 'eom_comparison', 'behaviour_profiles']:
    if chart_name in charts:
        show_chart(charts[chart_name], chart_name.replace('_', ' ').title())

if 'balance_trend' in charts:
    show_chart(charts['balance_trend'], 'Balance Trend')


## 9. ML Classification Results

In [ ]:
ml = result['ml_classification']
print(f"  Status: {ml.get('status', 'N/A')}  |  Mode: {ml.get('mode', 'N/A')}")
print()

if 'scores' in ml:
    for model_name, scores in ml['scores'].items():
        if 'cv_auc' in scores:
            print(f"  {model_name:<25}  CV-AUC: {scores['cv_auc']:.4f} ± {scores['cv_std']:.4f}  ({scores['method']})")
        elif 'auc' in scores:
            print(f"  {model_name:<25}  AUC: {scores['auc']:.4f}  ({scores['method']})")

if 'feature_importances' in ml:
    print("\n  Feature Importances (Random Forest):")
    for feat, imp in sorted(ml['feature_importances'].items(), key=lambda x: -x[1]):
        bar = '█' * int(imp * 40)
        print(f"    {feat:<35} {bar} {imp:.4f}")


## 10. Web Backend Integration Examples

In [ ]:
# ── Flask example ─────────────────────────────────────────────────────────────
flask_example = '''
from flask import Flask, request, jsonify
from bank_analyser import BankStatementAnalyser

app = Flask(__name__)
analyser = BankStatementAnalyser()   # initialise once at startup

@app.route('/analyse', methods=['POST'])
def analyse():
    if 'statement' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400

    f = request.files['statement']
    try:
        result = analyser.analyse(
            file_bytes=f.read(),
            filename=f.filename
        )
        return jsonify(result)
    except ValueError as e:
        return jsonify({'error': str(e)}), 422
    except Exception as e:
        return jsonify({'error': 'Analysis failed', 'detail': str(e)}), 500
'''

# ── FastAPI example ────────────────────────────────────────────────────────────
fastapi_example = '''
from fastapi import FastAPI, UploadFile, File, HTTPException
from bank_analyser import BankStatementAnalyser

app = FastAPI()
analyser = BankStatementAnalyser()   # initialise once at startup

@app.post("/analyse")
async def analyse(statement: UploadFile = File(...)):
    contents = await statement.read()
    try:
        result = analyser.analyse(file_bytes=contents, filename=statement.filename)
        return result
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
'''

print("Flask integration:")
print(flask_example)
print("\nFastAPI integration:")
print(fastapi_example)


## 11. Full JSON Output Structure (without chart bytes)

In [ ]:
# Show the full result structure without the large chart base64 strings
result_preview = {k: v for k, v in result.items() if k != 'charts'}
result_preview['charts'] = {k: '<base64 PNG string>' for k in result['charts']}
print(json.dumps(result_preview, indent=2, default=str))


---
## Summary

| Component | Detail |
|---|---|
| **Module** | `bank_analyser/analyser.py` — plug into Flask/FastAPI/Django |
| **Column detection** | Dynamic semantic mapping — works with any Indian bank |
| **Format support** | Dual-column, single-amount+flag, signed-amount, CSV/TSV/XLSX |
| **Risk scoring** | ML ensemble: Z-Score + Isolation Forest + PCA reconstruction error |
| **Thresholds** | Adaptive percentile-based — personalised per user |
| **Output** | Structured JSON with base64 charts — frontend-ready |

> **Submitted for:** Behavioural Analytics Hackathon — Problem Statement 2 | OrgX
